# DDPM on a 2D Toy Distribution / 2D 장난감 분포에서의 DDPM 구현

**Paper**: Ho, Jain, Abbeel (2020) — *Denoising Diffusion Probabilistic Models*

**한국어**
이 노트북은 DDPM의 핵심 알고리즘을 2D 장난감 분포(two moons / spirals)에 구현합니다. 목적은 픽셀 차원의 복잡함을 걷어내고 다음 4가지를 시각적으로 확인하는 것:
1. **Forward (diffusion) process** — 데이터에서 노이즈로의 점진적 손상
2. **Tiny $\epsilon$-predictor (MLP)** — `(x_t, t)` → `ε` 예측 신경망
3. **$L_\text{simple}$ 학습** — Algorithm 1을 그대로 구현
4. **Reverse sampling** — Algorithm 2를 그대로 구현, 노이즈에서 데이터로의 복원

**English**
This notebook implements DDPM's core algorithms on a 2D toy distribution (two moons / spirals). By stripping away pixel-domain complexity we can visually verify:
1. **Forward (diffusion) process** — gradual corruption from data to noise
2. **Tiny $\epsilon$-predictor (MLP)** — neural net mapping `(x_t, t)` → `ε`
3. **$L_\text{simple}$ training** — direct implementation of Algorithm 1
4. **Reverse sampling** — direct implementation of Algorithm 2, denoising from $\mathcal{N}(0, I)$ back to data

**Kernel**: `study-with-ai` conda environment (PyTorch).

**References**: Equations and algorithm numbers refer to Ho, Jain, Abbeel (2020), arXiv:2006.11239.

## 1. Setup and Toy Data / 설정 및 장난감 데이터

**한국어**
`make_moons` 데이터를 표준화하여 사용합니다. 2D 분포는 forward/reverse 과정을 직접 산점도로 확인할 수 있는 장점이 있습니다.

**English**
We use the `make_moons` dataset, standardized. A 2D distribution lets us visualize forward/reverse trajectories directly with scatter plots.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def sample_moons(n: int) -> torch.Tensor:
    """Sample n points from a standardized two-moons distribution.

    Args:
        n: Number of points to draw.

    Returns:
        Float tensor of shape (n, 2) with mean ~0 and unit-ish scale.
    """
    x, _ = make_moons(n_samples=n, noise=0.05, random_state=0)
    x = (x - x.mean(axis=0)) / x.std(axis=0)
    return torch.tensor(x, dtype=torch.float32)


data = sample_moons(4000)
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.scatter(data[:, 0], data[:, 1], s=2, alpha=0.6)
ax.set_title("Two Moons (data $x_0$) / 데이터 분포")
ax.set_aspect("equal")
plt.show()

## 2. $\beta$ Schedule and Closed-Form Forward Process / $\beta$ 스케줄과 닫힌 형태의 순방향 과정

**한국어**
DDPM은 $T=1000$, $\beta_1 = 10^{-4} \to \beta_T = 0.02$ 선형 스케줄을 사용합니다. 토이 문제에서는 $T=200$로 줄여 학습을 빠르게 합니다. 닫힌 형태(식 4)를 활용해 임의의 $t$에서 한 번에 $x_t$를 샘플링합니다:
$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

**English**
The original paper uses $T=1000$ with linear $\beta$ from $10^{-4}$ to $0.02$. For this toy problem we use $T=200$ to keep training fast. We exploit the closed-form Eq. (4) to sample $x_t$ in one step:
$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$$

In [ ]:
T = 200
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alphas_bar = torch.cumprod(alphas, dim=0)
sqrt_alphas_bar = torch.sqrt(alphas_bar)
sqrt_one_minus_alphas_bar = torch.sqrt(1.0 - alphas_bar)


def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
    """Sample x_t in closed form from Eq. (4) of Ho et al. (2020).

    Args:
        x0: Clean data tensor of shape (B, D).
        t: Long tensor of timesteps in [0, T-1] of shape (B,).
        noise: Pre-sampled standard normal noise matching x0's shape.

    Returns:
        Noised tensor x_t of shape (B, D).
    """
    sa = sqrt_alphas_bar[t].unsqueeze(-1)
    soma = sqrt_one_minus_alphas_bar[t].unsqueeze(-1)
    return sa * x0 + soma * noise


# Visualize forward process at several t / 여러 t에서 순방향 과정 시각화
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
snapshot_ts = [0, T // 8, T // 4, T // 2, T - 1]
for ax, ti in zip(axes, snapshot_ts):
    x0 = data.to(device)
    eps = torch.randn_like(x0)
    t_tensor = torch.full((x0.shape[0],), ti, dtype=torch.long, device=device)
    xt = q_sample(x0, t_tensor, eps).cpu()
    ax.scatter(xt[:, 0], xt[:, 1], s=2, alpha=0.5)
    ax.set_title(f"$x_t$ at t={ti}")
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
fig.suptitle("Forward diffusion process / 순방향 확산 과정")
plt.tight_layout()
plt.show()

## 3. Tiny $\epsilon$-Predictor (MLP with Sinusoidal Time Embedding) / 작은 노이즈 예측기

**한국어**
원 논문은 U-Net 백본을 쓰지만 2D 데이터에는 과합니다. 우리는 MLP $\epsilon$-predictor에 **Transformer 식 sinusoidal time embedding** (논문 §4)을 그대로 도입합니다. 입력: $x_t \in \mathbb{R}^2$와 임베딩된 시간 $t$. 출력: 예측 노이즈 $\hat\epsilon \in \mathbb{R}^2$.

**English**
The original paper uses a U-Net backbone, but that's overkill for 2D. We use an MLP $\epsilon$-predictor with **Transformer-style sinusoidal time embedding** (paper §4). Input: $x_t \in \mathbb{R}^2$ and embedded $t$. Output: predicted noise $\hat\epsilon \in \mathbb{R}^2$.

In [ ]:
def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """Compute Transformer-style sinusoidal positional embedding for timestep t.

    Args:
        t: Long tensor of timesteps of shape (B,).
        dim: Embedding dimensionality (must be even).

    Returns:
        Float tensor of shape (B, dim) with interleaved sin/cos features.
    """
    half = dim // 2
    freqs = torch.exp(
        -np.log(10000.0) * torch.arange(half, dtype=torch.float32, device=t.device) / half
    )
    args = t.float().unsqueeze(-1) * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class EpsPredictor(nn.Module):
    """Tiny MLP that predicts the noise epsilon added to x_t at timestep t.

    Architecture: time-embed -> MLP || x_t -> MLP -> output. Time embedding is
    injected via concatenation in each hidden block, mirroring the residual-block
    injection used in DDPM's U-Net.
    """

    def __init__(self, data_dim: int = 2, hidden: int = 256, time_dim: int = 64):
        super().__init__()
        self.time_dim = time_dim
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
        )
        self.in_proj = nn.Linear(data_dim, hidden)
        self.block1 = nn.Sequential(nn.SiLU(), nn.Linear(hidden, hidden))
        self.block2 = nn.Sequential(nn.SiLU(), nn.Linear(hidden, hidden))
        self.block3 = nn.Sequential(nn.SiLU(), nn.Linear(hidden, hidden))
        self.out = nn.Linear(hidden, data_dim)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict epsilon given noised input x_t and timestep t.

        Args:
            x: Noised data x_t of shape (B, data_dim).
            t: Long tensor of timesteps of shape (B,).

        Returns:
            Predicted noise of shape (B, data_dim).
        """
        emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))
        h = self.in_proj(x) + emb
        h = h + self.block1(h)
        h = h + self.block2(h)
        h = h + self.block3(h)
        return self.out(h)


model = EpsPredictor().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## 4. Training Loop ($L_\text{simple}$, Algorithm 1) / 학습 루프

**한국어**
Algorithm 1을 그대로 구현합니다 — `t`와 `ε`를 무작위로 뽑고 $L_\text{simple} = \|\epsilon - \epsilon_\theta(x_t, t)\|^2$의 gradient step을 반복합니다.

**English**
Direct implementation of Algorithm 1 — sample random `t` and `ε`, take gradient steps on $L_\text{simple} = \|\epsilon - \epsilon_\theta(x_t, t)\|^2$.

In [ ]:
def train(model: nn.Module, n_steps: int = 5000, batch_size: int = 256) -> list:
    """Train the epsilon predictor on the simplified DDPM loss (Eq. 14).

    Implements Algorithm 1 of Ho et al. (2020): sample x_0, sample t uniformly,
    sample epsilon ~ N(0, I), and take a gradient step on ||eps - eps_theta||^2.

    Args:
        model: Epsilon predictor network.
        n_steps: Number of gradient steps.
        batch_size: Batch size per step.

    Returns:
        List of training losses (one per step).
    """
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)
    losses = []
    model.train()
    for step in range(n_steps):
        x0 = sample_moons(batch_size).to(device)
        t = torch.randint(0, T, (batch_size,), device=device)
        eps = torch.randn_like(x0)
        xt = q_sample(x0, t, eps)
        eps_pred = model(xt, t)
        loss = F.mse_loss(eps_pred, eps)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if (step + 1) % 1000 == 0:
            print(f"step {step + 1:5d}/{n_steps}  loss = {np.mean(losses[-100:]):.4f}")
    return losses


losses = train(model, n_steps=5000, batch_size=256)

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(np.convolve(losses, np.ones(50) / 50, mode="valid"))
ax.set_xlabel("Step")
ax.set_ylabel("$L_\\text{simple}$ (smoothed)")
ax.set_title("Training loss / 학습 손실")
plt.tight_layout()
plt.show()

## 5. Reverse Sampling (Algorithm 2) / 역방향 샘플링

**한국어**
Algorithm 2를 그대로 구현. $x_T \sim \mathcal{N}(0, I)$에서 시작하여 $t = T, \ldots, 1$로 내려가며
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t, t)\right) + \sigma_t z$$
$z \sim \mathcal{N}(0, I)$ if $t > 1$, else $z = 0$. 우리는 $\sigma_t^2 = \beta_t$를 사용 (논문이 권장하는 두 옵션 중 하나).

**English**
Direct implementation of Algorithm 2. Starting from $x_T \sim \mathcal{N}(0, I)$, iterate from $t = T$ down to $1$ using the formula above with $\sigma_t^2 = \beta_t$ (one of the two paper-recommended options).

In [ ]:
@torch.no_grad()
def sample(model: nn.Module, n: int, save_every: int = 20) -> tuple:
    """Generate samples by running Algorithm 2 of Ho et al. (2020).

    Args:
        model: Trained epsilon predictor.
        n: Number of samples to draw.
        save_every: Save x_t every this many reverse steps for visualization.

    Returns:
        Tuple of (final samples x_0, list of intermediate (t, x_t) snapshots).
    """
    model.eval()
    x = torch.randn(n, 2, device=device)
    snapshots = [(T, x.cpu().clone())]
    for ti in reversed(range(T)):
        t_tensor = torch.full((n,), ti, dtype=torch.long, device=device)
        eps_pred = model(x, t_tensor)
        beta_t = betas[ti]
        alpha_t = alphas[ti]
        alpha_bar_t = alphas_bar[ti]
        coef = beta_t / torch.sqrt(1.0 - alpha_bar_t)
        mean = (1.0 / torch.sqrt(alpha_t)) * (x - coef * eps_pred)
        if ti > 0:
            sigma_t = torch.sqrt(beta_t)
            z = torch.randn_like(x)
            x = mean + sigma_t * z
        else:
            x = mean
        if ti % save_every == 0 or ti == 0:
            snapshots.append((ti, x.cpu().clone()))
    return x.cpu(), snapshots


samples, snaps = sample(model, n=4000, save_every=20)

fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
ax[0].scatter(data[:, 0], data[:, 1], s=2, alpha=0.5)
ax[0].set_title("True data / 실제 데이터 ($x_0$)")
ax[0].set_aspect("equal")
ax[0].set_xlim(-3, 3)
ax[0].set_ylim(-3, 3)
ax[1].scatter(samples[:, 0], samples[:, 1], s=2, alpha=0.5, color="crimson")
ax[1].set_title("DDPM samples / DDPM 샘플")
ax[1].set_aspect("equal")
ax[1].set_xlim(-3, 3)
ax[1].set_ylim(-3, 3)
plt.tight_layout()
plt.show()

## 6. Reverse Trajectory Visualization / 역방향 궤적 시각화

**한국어**
노이즈 $x_T$에서 데이터 $x_0$로 가는 reverse process의 중간 단계를 산점도 그리드로 보여줍니다. 큰 $t$에서 작은 $t$로 갈수록 분포가 점점 두 초승달 모양으로 정돈됩니다.

**English**
Grid of scatter plots showing intermediate $x_t$ along the reverse process from noise $x_T$ to data $x_0$. As $t$ decreases, the distribution progressively organizes into the two-moons shape.

In [ ]:
# Pick 8 evenly spaced snapshots from the reverse trajectory.
# 역방향 궤적에서 8개 균등 간격 스냅샷 선택.
snaps_sorted = sorted(snaps, key=lambda p: -p[0])  # descending t
step = max(1, len(snaps_sorted) // 8)
selected = snaps_sorted[::step][:8]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (ti, xt) in zip(axes.flatten(), selected):
    ax.scatter(xt[:, 0], xt[:, 1], s=2, alpha=0.5, color="steelblue")
    ax.set_title(f"t = {ti}")
    ax.set_aspect("equal")
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
fig.suptitle("Reverse process: $x_T \\to x_0$ / 역방향 과정")
plt.tight_layout()
plt.show()

## 7. Quantitative Sanity Check / 정량 검증

**한국어**
생성 샘플과 실제 데이터의 기초 통계량(평균, 공분산)과 2-Wasserstein 근사(샘플 평균/표준편차 차이)를 비교합니다. 완벽한 일치를 기대하는 것은 아니지만, 합리적 범위 내인지 확인.

**English**
Compare basic statistics (mean, covariance) of generated samples vs. real data, plus a coarse 2-Wasserstein-style sanity check via mean/std differences. Not a perfect match — just a reasonable-range sanity check.

In [ ]:
real = data.numpy()
gen = samples.numpy()

print("== Mean / 평균 ==")
print(f"  real: {real.mean(axis=0)}")
print(f"  gen : {gen.mean(axis=0)}")

print("\n== Std / 표준편차 ==")
print(f"  real: {real.std(axis=0)}")
print(f"  gen : {gen.std(axis=0)}")

print("\n== Covariance / 공분산 행렬 ==")
print("real:")
print(np.cov(real.T))
print("gen:")
print(np.cov(gen.T))

# Histogram comparison along x-axis / x축 히스토그램 비교
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, dim, name in zip(axes, [0, 1], ["x", "y"]):
    ax.hist(real[:, dim], bins=60, alpha=0.5, label="real", density=True)
    ax.hist(gen[:, dim], bins=60, alpha=0.5, label="gen", density=True, color="crimson")
    ax.set_title(f"Marginal of {name} / {name} 주변 분포")
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Summary / 요약

**한국어**
DDPM의 핵심 11줄(Algorithm 1 학습 6줄 + Algorithm 2 샘플링 5줄)을 그대로 2D 토이 데이터에 적용하여 다음을 확인했습니다:

1. **닫힌 형태 forward sampling** (식 4)이 임의의 $t$에서 $x_t$를 한 번에 만든다.
2. **노이즈 예측($\epsilon$-prediction) + $L_\text{simple}$** 손실로 작은 MLP가 빠르게 학습된다.
3. **Sinusoidal time embedding**이 단일 모델로 $T$개의 노이즈 스케일을 모두 처리하게 해준다.
4. **Algorithm 2 reverse sampling**이 $\mathcal{N}(0, I)$에서 시작해 두 초승달 분포를 정확히 복원한다.

이 11줄이 Stable Diffusion, DALL·E 2, Imagen, Sora의 직접적 토대입니다.

**English**
We applied the 11 core lines of DDPM (6-line Algorithm 1 training + 5-line Algorithm 2 sampling) directly to a 2D toy distribution, verifying that:

1. **Closed-form forward sampling** (Eq. 4) generates $x_t$ at any $t$ in a single shot.
2. **$\epsilon$-prediction with $L_\text{simple}$** trains a small MLP to convergence very quickly.
3. **Sinusoidal time embedding** lets a single network handle all $T$ noise scales.
4. **Algorithm 2 reverse sampling** starts from $\mathcal{N}(0, I)$ and faithfully reconstructs the two-moons distribution.

These 11 lines are the direct foundation of Stable Diffusion, DALL·E 2, Imagen, and Sora.